### End-to-End Pipeline Evaluation — OmniDocBench JASyn

So sánh đối đầu 3 pipeline (**`default_docling`**, **`plugged_docling`**, **`custom_parser`**) trên **518 trang OmniDocBench JASyn** (tiếng Nhật) qua **5 khía cạnh**.

1. **Layout Detection:** `F1@0.50`, `mAP@0.50:0.95` (9 lớp nhãn chuẩn, IoU `0.50:0.95`).
2. **Text / OCR:** `NED` *(càng thấp càng tốt)*, `Similarity (1 - NED)` *(càng cao càng tốt)*, `Micro CER`, `Len Ratio`.
3. **Reading Order:** `Kendall's Tau (τ)` *(`[-1, 1]`, càng gần `1.0` càng đúng thứ tự đọc)*, `NID` *(Normalized Indel Distance = `1 - LCS/k`, càng gần `0.0` càng tốt)*.
4. **Table Structure:** `TEDS`, `TEDS-Struct` *(exact 3-level Zhang-Shasha)*, `Match %` *(IoU ≥ 0.1)*.
5. **Hiệu năng hệ thống:** `Avg s/trang` *(1 GPU worker)*, `Wall s/trang` *(song song 2 GPU)*, `Peak RAM (MB/worker)`, `Peak VRAM (MB/GPU)`.

In [1]:
import bisect
import json
import re
import unicodedata
from collections import defaultdict
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from lxml import html as lxml_html

BASE_DIR = Path(".").resolve()
GT_PATH = BASE_DIR / "ground_truth.json"
SYSTEMS = ["default_docling", "plugged_docling", "custom_parser"]
IS_JAPANESE = True
PDF_TYPE_FIELD = 'layout'
EXCLUDE_GT_TEXT_CATS = set()
EXCLUDE_PRED_TEXT_LABELS = set()
META_FILES = {"errors.json", "current_files.json", "perf.json", "ground_truth.json"}

IOU_THRESHOLDS = np.linspace(0.50, 0.95, 10)
REC_THRESHOLDS = np.linspace(0.0, 1.0, 101)
MAX_DETS = 100

OMNI_GT_TO_COMMON = {
    "text_block": "text",
    "reference": "text",
    "code_txt": "text",
    "equation_explanation": "text",
    "list_group": "text",
    "title": "section_header",
    "table": "table",
    "figure": "picture",
    "figure_caption": "caption",
    "table_caption": "caption",
    "code_txt_caption": "caption",
    "equation_caption": "caption",
    "equation_isolated": "formula",
    "equation_semantic": "formula",
    "header": "page_header",
    "footer": "page_footer",
    "page_number": "page_footer",
    "page_footnote": "footnote",
    "figure_footnote": "footnote",
    "table_footnote": "footnote",
}

PRED_LABEL_TO_COMMON = {
    "text": "text",
    "paragraph": "text",
    "code": "text",
    "reference": "text",
    "list_item": "text",
    "form": "text",
    "key_value_region": "text",
    "section_header": "section_header",
    "title": "section_header",
    "heading": "section_header",
    "table": "table",
    "document_index": "table",
    "picture": "picture",
    "figure": "picture",
    "chart": "picture",
    "caption": "caption",
    "formula": "formula",
    "page_header": "page_header",
    "page_footer": "page_footer",
    "footnote": "footnote",
}


def is_custom_parser(doc: dict) -> bool:
    return "elements" in doc and "texts" not in doc


# --- 1. Layout Helpers ---
def compute_iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float64)
    x1 = np.maximum(boxes_a[:, None, 0], boxes_b[None, :, 0])
    y1 = np.maximum(boxes_a[:, None, 1], boxes_b[None, :, 1])
    x2 = np.minimum(boxes_a[:, None, 2], boxes_b[None, :, 2])
    y2 = np.minimum(boxes_a[:, None, 3], boxes_b[None, :, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_a = np.maximum(0.0, boxes_a[:, 2] - boxes_a[:, 0]) * np.maximum(0.0, boxes_a[:, 3] - boxes_a[:, 1])
    area_b = np.maximum(0.0, boxes_b[:, 2] - boxes_b[:, 0]) * np.maximum(0.0, boxes_b[:, 3] - boxes_b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0.0, inter / union, 0.0)


def _docling_page_size(doc: dict) -> tuple[float, float]:
    pages = doc.get("pages") or {}
    pinfo = pages.get("1") or (next(iter(pages.values())) if pages else None)
    if not pinfo or "size" not in pinfo:
        return 0.0, 0.0
    return float(pinfo["size"]["width"]), float(pinfo["size"]["height"])


def _custom_page_size(doc: dict) -> tuple[float, float]:
    pages = (doc.get("metadata") or {}).get("pages") or []
    if not pages:
        return 0.0, 0.0
    return float(pages[0].get("width_pt", 0.0)), float(pages[0].get("height_pt", 0.0))


def _docling_item_box_px(item: dict, page_w: float, page_h: float, gt_w: float, gt_h: float):
    provs = item.get("prov") or ()
    if not provs:
        return None
    bb = provs[0].get("bbox")
    if not bb:
        return None
    l, t, r, b = float(bb["l"]), float(bb["t"]), float(bb["r"]), float(bb["b"])
    origin = str(bb.get("coord_origin", "BOTTOMLEFT")).upper()
    y1, y2 = (page_h - t, page_h - b) if origin == "BOTTOMLEFT" else (t, b)
    sx, sy = gt_w / page_w, gt_h / page_h
    x1, x2 = min(l, r) * sx, max(l, r) * sx
    y1, y2 = min(y1, y2) * sy, max(y1, y2) * sy
    x1, x2 = min(max(x1, 0.0), gt_w), min(max(x2, 0.0), gt_w)
    y1, y2 = min(max(y1, 0.0), gt_h), min(max(y2, 0.0), gt_h)
    return [x1, y1, x2, y2] if (x2 > x1 and y2 > y1) else None


def _custom_box_px(bb, page_w: float, page_h: float, gt_w: float, gt_h: float):
    if not bb or len(bb) != 4:
        return None
    sx, sy = gt_w / page_w, gt_h / page_h
    x1, y1, x2, y2 = float(bb[0]) * sx, float(bb[1]) * sy, float(bb[2]) * sx, float(bb[3]) * sy
    x1, x2 = min(max(min(x1, x2), 0.0), gt_w), min(max(max(x1, x2), 0.0), gt_w)
    y1, y2 = min(max(min(y1, y2), 0.0), gt_h), min(max(max(y1, y2), 0.0), gt_h)
    return [x1, y1, x2, y2] if (x2 > x1 and y2 > y1) else None


def extract_pred_layout_boxes(doc: dict, gt_w: float, gt_h: float) -> dict[str, list[list[float]]]:
    by_cat: dict[str, list[list[float]]] = defaultdict(list)
    if is_custom_parser(doc):
        pw, ph = _custom_page_size(doc)
        if pw <= 0 or ph <= 0:
            return by_cat
        for el in doc.get("elements") or ():
            if el.get("level", 1) != 1:
                continue
            meta = el.get("metadata") or {}
            raw_label = str(meta.get("layout_label") or meta.get("role") or el.get("type") or "").strip().lower()
            mapped = PRED_LABEL_TO_COMMON.get(raw_label)
            if mapped is None:
                continue
            box = _custom_box_px(el.get("bbox"), pw, ph, gt_w, gt_h)
            if box is not None:
                by_cat[mapped].append(box)
        return by_cat

    pw, ph = _docling_page_size(doc)
    if pw <= 0 or ph <= 0:
        return by_cat
    for section in ("texts", "tables", "pictures", "key_value_items"):
        for item in doc.get(section) or ():
            raw_label = str(item.get("label", "")).strip().lower()
            mapped = PRED_LABEL_TO_COMMON.get(raw_label)
            if mapped is None:
                continue
            box = _docling_item_box_px(item, pw, ph, gt_w, gt_h)
            if box is not None:
                by_cat[mapped].append(box)
    return by_cat


def compute_ap_101(tp_flags: np.ndarray, total_gt: int) -> float:
    if total_gt == 0:
        return float("nan")
    if len(tp_flags) == 0:
        return 0.0
    tp_cum = np.cumsum(tp_flags, dtype=np.float64)
    fp_cum = np.cumsum(1.0 - tp_flags, dtype=np.float64)
    rec = tp_cum / total_gt
    prec = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)
    for i in range(len(prec) - 1, 0, -1):
        if prec[i] > prec[i - 1]:
            prec[i - 1] = prec[i]
    inds = np.searchsorted(rec, REC_THRESHOLDS, side="left")
    q = np.zeros(len(REC_THRESHOLDS), dtype=np.float64)
    valid = inds < len(prec)
    q[valid] = prec[inds[valid]]
    return float(np.mean(q))


def prf1(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


def evaluate_layout_subset(image_ids, gt_by_img_cat, pred_by_img_cat, cat_names):
    n_t = len(IOU_THRESHOLDS)
    idx_50, idx_75 = 0, 5
    per_cat_ap, per_cat_f1_50 = {}, {}
    total_tp_50 = total_fp_50 = total_fn_50 = 0
    total_tp_75 = total_fp_75 = total_fn_75 = 0

    for cname in cat_names:
        tp_lists = [[] for _ in range(n_t)]
        total_gt = 0
        cat_tp_50 = cat_fp_50 = cat_tp_75 = cat_fp_75 = 0
        for img_id in image_ids:
            gt_boxes = gt_by_img_cat.get((img_id, cname))
            pred_boxes = pred_by_img_cat.get((img_id, cname))
            n_gt = 0 if gt_boxes is None else len(gt_boxes)
            n_pred = 0 if pred_boxes is None else min(len(pred_boxes), MAX_DETS)
            total_gt += n_gt
            if n_pred == 0:
                continue
            if n_gt == 0:
                for t_idx in range(n_t):
                    tp_lists[t_idx].extend([0.0] * n_pred)
                cat_fp_50 += n_pred
                cat_fp_75 += n_pred
                continue
            ious = compute_iou_matrix(pred_boxes[:n_pred], gt_boxes)
            for t_idx, thr in enumerate(IOU_THRESHOLDS):
                matched_gt = np.zeros(n_gt, dtype=bool)
                tp_img = 0
                for d_idx in range(n_pred):
                    row = ious[d_idx]
                    best_g, best_iou = -1, thr - 1e-12
                    for g_idx in range(n_gt):
                        if not matched_gt[g_idx] and row[g_idx] >= best_iou:
                            best_iou = row[g_idx]
                            best_g = g_idx
                    if best_g >= 0:
                        matched_gt[best_g] = True
                        tp_lists[t_idx].append(1.0)
                        tp_img += 1
                    else:
                        tp_lists[t_idx].append(0.0)
                if t_idx == idx_50:
                    cat_tp_50 += tp_img
                    cat_fp_50 += n_pred - tp_img
                elif t_idx == idx_75:
                    cat_tp_75 += tp_img
                    cat_fp_75 += n_pred - tp_img
        aps = [compute_ap_101(np.asarray(tp_lists[t_idx], dtype=np.float64), total_gt) for t_idx in range(n_t)]
        per_cat_ap[cname] = {
            "mAP_50_95": float(np.nanmean(aps)) if not all(np.isnan(aps)) else float("nan"),
            "AP_50": aps[idx_50],
            "AP_75": aps[idx_75],
            "gt_count": total_gt,
        }
        _, _, per_cat_f1_50[cname] = prf1(cat_tp_50, cat_fp_50, total_gt - cat_tp_50)
        total_tp_50 += cat_tp_50
        total_fp_50 += cat_fp_50
        total_fn_50 += total_gt - cat_tp_50
        total_tp_75 += cat_tp_75
        total_fp_75 += cat_fp_75
        total_fn_75 += total_gt - cat_tp_75

    valid_map = [v["mAP_50_95"] for v in per_cat_ap.values() if not np.isnan(v["mAP_50_95"])]
    valid_ap50 = [v["AP_50"] for v in per_cat_ap.values() if not np.isnan(v["AP_50"])]
    valid_ap75 = [v["AP_75"] for v in per_cat_ap.values() if not np.isnan(v["AP_75"])]
    p50, r50, f1_50 = prf1(total_tp_50, total_fp_50, total_fn_50)
    _, _, f1_75 = prf1(total_tp_75, total_fp_75, total_fn_75)
    return {
        "mAP_50_95": float(np.mean(valid_map)) if valid_map else 0.0,
        "AP_50": float(np.mean(valid_ap50)) if valid_ap50 else 0.0,
        "AP_75": float(np.mean(valid_ap75)) if valid_ap75 else 0.0,
        "P_50": p50,
        "R_50": r50,
        "F1_50": f1_50,
        "F1_75": f1_75,
        "per_cat_ap": per_cat_ap,
        "per_cat_f1_50": per_cat_f1_50,
    }


# --- 2. Text / OCR Helpers ---
def levenshtein_distance(a: str, b: str) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    if len(a) > len(b):
        a, b = b, a
    m = len(a)
    peq: dict[str, int] = {}
    for i, ch in enumerate(a):
        peq[ch] = peq.get(ch, 0) | (1 << i)
    pv = (1 << m) - 1
    mv = 0
    score = m
    hb = 1 << (m - 1)
    mask = (1 << m) - 1
    for ch in b:
        eq = peq.get(ch, 0)
        xv = eq | mv
        xh = (((eq & pv) + pv) ^ pv) | eq
        ph = mv | (~(xh | pv) & mask)
        mh = pv & xh
        if ph & hb:
            score += 1
        elif mh & hb:
            score -= 1
        ph = ((ph << 1) | 1) & mask
        mh = (mh << 1) & mask
        pv = mh | (~(xv | ph) & mask)
        mv = ph & xv
    return score


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    if IS_JAPANESE:
        return re.sub(r"\s+", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _iter_docling_ordered_texts(doc: dict) -> list[dict]:
    texts = doc.get("texts") or []
    groups = doc.get("groups") or []
    body = doc.get("body") or {}
    ordered = []
    seen = set()

    def _walk(children):
        for ch in children or ():
            ref = ch.get("$ref", "")
            if ref.startswith("#/texts/"):
                idx = int(ref.split("/")[-1])
                if 0 <= idx < len(texts) and idx not in seen:
                    seen.add(idx)
                    ordered.append(texts[idx])
            elif ref.startswith("#/groups/"):
                idx = int(ref.split("/")[-1])
                if 0 <= idx < len(groups):
                    _walk(groups[idx].get("children"))

    _walk(body.get("children"))
    if not ordered:
        return list(texts)
    return ordered


def extract_gt_ordered_text_blocks(page: dict) -> list[tuple[list[float], str]]:
    items = []
    for d in page.get("layout_dets", []):
        if d.get("ignore"):
            continue
        if d.get("category_type") in EXCLUDE_GT_TEXT_CATS:
            continue
        txt = (d.get("text") or "").strip()
        if not txt:
            continue
        poly = d.get("poly")
        if poly and len(poly) >= 8:
            xs = [float(poly[i]) for i in range(0, len(poly), 2)]
            ys = [float(poly[i]) for i in range(1, len(poly), 2)]
            box = [min(xs), min(ys), max(xs), max(ys)]
        elif d.get("bbox") and len(d["bbox"]) == 4:
            x, y, w, h = [float(v) for v in d["bbox"]]
            box = [x, y, x + w, y + h]
        else:
            box = [0.0, 0.0, 0.0, 0.0]
        order = d.get("order")
        key = (0, order) if order is not None else (1, box[1], box[0])
        items.append((key, order is not None, box, txt))
    items.sort(key=lambda x: x[0])
    return items


def extract_pred_ordered_text_blocks(doc: dict, gt_w: float, gt_h: float) -> list[tuple[list[float], str]]:
    out = []
    if is_custom_parser(doc):
        pw, ph = _custom_page_size(doc)
        elems = [
            el for el in (doc.get("elements") or ())
            if el.get("level", 1) == 1 and el.get("type") == "text"
        ]
        elems.sort(key=lambda el: el.get("reading_order", 10**9))
        for el in elems:
            meta = el.get("metadata") or {}
            lbl = str(meta.get("layout_label") or meta.get("role") or "").strip().lower()
            if lbl in EXCLUDE_PRED_TEXT_LABELS:
                continue
            txt = (el.get("text") or "").strip()
            if not txt:
                continue
            box = _custom_box_px(el.get("bbox"), pw, ph, gt_w, gt_h) if (pw > 0 and ph > 0) else None
            out.append((box or [0.0, 0.0, 0.0, 0.0], txt))
        return out

    pw, ph = _docling_page_size(doc)
    for t in _iter_docling_ordered_texts(doc):
        lbl = str(t.get("label", "")).strip().lower()
        if lbl in EXCLUDE_PRED_TEXT_LABELS:
            continue
        txt = (t.get("text") or "").strip()
        if not txt:
            continue
        box = _docling_item_box_px(t, pw, ph, gt_w, gt_h) if (pw > 0 and ph > 0) else None
        out.append((box or [0.0, 0.0, 0.0, 0.0], txt))
    return out


# --- 3. Reading Order Helpers ---
def kendall_tau_from_ranks(ranks: list[int]) -> float:
    k = len(ranks)
    if k < 2:
        return float("nan")
    concordant = discordant = 0
    for i in range(k):
        ri = ranks[i]
        for j in range(i + 1, k):
            if ri < ranks[j]:
                concordant += 1
            elif ri > ranks[j]:
                discordant += 1
    total = k * (k - 1) / 2.0
    return (concordant - discordant) / total if total > 0 else float("nan")


def nid_from_ranks(ranks: list[int]) -> float:
    """Normalized Indel Distance = 1 - LIS(ranks) / len(ranks) in [0, 1] (lower is better)."""
    k = len(ranks)
    if k < 2:
        return float("nan")
    tails: list[int] = []
    for r in ranks:
        pos = bisect.bisect_left(tails, r)
        if pos == len(tails):
            tails.append(r)
        else:
            tails[pos] = r
    return 1.0 - (len(tails) / k)


def evaluate_reading_order_page(gt_blocks, pred_blocks):
    # Only use GT blocks that have explicit `order` and valid bbox
    gt_ord = [(b, t) for _, has_ord, b, t in gt_blocks if has_ord and b[2] > b[0] and b[3] > b[1]]
    if len(gt_ord) < 2:
        return None
    pr_valid = [(idx, b) for idx, (b, _) in enumerate(pred_blocks) if b[2] > b[0] and b[3] > b[1]]
    if not pr_valid:
        return {"gt_blocks": len(gt_ord), "matched": 0, "tau": float("nan"), "nid": float("nan")}

    gt_arr = np.asarray([b for b, _ in gt_ord], dtype=np.float64)
    pr_arr = np.asarray([b for _, b in pr_valid], dtype=np.float64)
    ious = compute_iou_matrix(gt_arr, pr_arr)

    # Greedy 1-to-1 matching by IoU >= 0.1
    pairs = []
    for gi in range(len(gt_ord)):
        for pj in range(len(pr_valid)):
            if ious[gi, pj] >= 0.1:
                pairs.append((float(ious[gi, pj]), gi, pj))
    pairs.sort(reverse=True)
    used_g, used_p = set(), set()
    g_to_pred_rank: dict[int, int] = {}
    for _, gi, pj in pairs:
        if gi in used_g or pj in used_p:
            continue
        used_g.add(gi)
        used_p.add(pj)
        g_to_pred_rank[gi] = pr_valid[pj][0]

    matched_ranks = [g_to_pred_rank[gi] for gi in sorted(g_to_pred_rank)]
    return {
        "gt_blocks": len(gt_ord),
        "matched": len(matched_ranks),
        "tau": kendall_tau_from_ranks(matched_ranks),
        "nid": nid_from_ranks(matched_ranks),
    }


# --- 4. Table Structure Helpers (exact 3-level Zhang-Shasha TEDS) ---
def norm_cell_text(s: str) -> str:
    s = unicodedata.normalize("NFKC", s or "")
    return re.sub(r"\s+", " ", s).strip()


class Node:
    __slots__ = ("tag", "colspan", "rowspan", "text", "children")

    def __init__(self, tag, colspan=1, rowspan=1, text="", children=None):
        self.tag = tag
        self.colspan = colspan
        self.rowspan = rowspan
        self.text = text
        self.children = children if children is not None else []


def html_to_tree(html_str: str) -> Node:
    if not html_str or not html_str.strip():
        return Node("table")
    try:
        doc = lxml_html.fromstring(html_str)
    except Exception:
        return Node("table")
    tables = doc.xpath(".//table") if doc.tag.lower() != "table" else [doc]
    tbl = tables[0] if tables else doc
    rows = []
    for tr in tbl.xpath(".//tr"):
        cells = []
        for cell in tr.xpath("./td | ./th"):
            try:
                cs = max(1, int(cell.attrib.get("colspan", 1)))
            except Exception:
                cs = 1
            try:
                rs = max(1, int(cell.attrib.get("rowspan", 1)))
            except Exception:
                rs = 1
            txt = norm_cell_text("".join(cell.itertext()))
            cells.append(Node("td", cs, rs, txt))
        if cells:
            rows.append(Node("tr", children=cells))
    return Node("table", children=rows)


def docling_table_to_tree(tbl: dict) -> Node:
    data = tbl.get("data") or {}
    cells = data.get("table_cells") or []
    if not cells:
        return Node("table")
    by_row: dict[int, list[tuple[int, Node]]] = {}
    for c in cells:
        r0 = int(c.get("start_row_offset_idx", 0))
        c0 = int(c.get("start_col_offset_idx", 0))
        rs = max(1, int(c.get("row_span", 1)))
        cs = max(1, int(c.get("col_span", 1)))
        txt = norm_cell_text(c.get("text", ""))
        by_row.setdefault(r0, []).append((c0, Node("td", cs, rs, txt)))
    rows = []
    for r_idx in sorted(by_row):
        row_cells = [n for _, n in sorted(by_row[r_idx], key=lambda x: x[0])]
        rows.append(Node("tr", children=row_cells))
    return Node("table", children=rows)


def is_complex_tree(root: Node) -> bool:
    return any(c.colspan > 1 or c.rowspan > 1 for tr in root.children for c in tr.children)


def tree_size(root: Node) -> int:
    return 1 + sum(1 + len(tr.children) for tr in root.children)


def cell_rename_cost(a: Node, b: Node, struct_only: bool) -> float:
    if a.colspan != b.colspan or a.rowspan != b.rowspan:
        return 1.0
    if struct_only or a.text == b.text:
        return 0.0
    denom = max(len(a.text), len(b.text))
    return levenshtein_distance(a.text, b.text) / denom if denom > 0 else 0.0


def row_ted(ra: Node, rb: Node, struct_only: bool) -> float:
    ca, cb = ra.children, rb.children
    m, n = len(ca), len(cb)
    prev = [float(j) for j in range(n + 1)]
    for i in range(1, m + 1):
        curr = [float(i)] + [0.0] * n
        ai = ca[i - 1]
        for j in range(1, n + 1):
            cost = cell_rename_cost(ai, cb[j - 1], struct_only)
            curr[j] = min(prev[j] + 1.0, curr[j - 1] + 1.0, prev[j - 1] + cost)
        prev = curr
    return prev[n]


def compute_teds(pred: Node, gt: Node, struct_only: bool = False) -> float:
    sa, sb = tree_size(pred), tree_size(gt)
    if sa == 1 and sb == 1:
        return 1.0
    if sa == 1 or sb == 1:
        return 0.0
    ra, rb = pred.children, gt.children
    m, n = len(ra), len(rb)
    prev = [0.0] * (n + 1)
    for j in range(1, n + 1):
        prev[j] = prev[j - 1] + 1.0 + len(rb[j - 1].children)
    for i in range(1, m + 1):
        del_cost = 1.0 + len(ra[i - 1].children)
        curr = [prev[0] + del_cost] + [0.0] * n
        rai = ra[i - 1]
        for j in range(1, n + 1):
            ins_cost = 1.0 + len(rb[j - 1].children)
            sub_cost = row_ted(rai, rb[j - 1], struct_only)
            curr[j] = min(prev[j] + del_cost, curr[j - 1] + ins_cost, prev[j - 1] + sub_cost)
        prev = curr
    return max(0.0, 1.0 - prev[n] / max(sa, sb))


def box_iou(b1, b2) -> float:
    ix0, iy0 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix1, iy1 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    if inter <= 0:
        return 0.0
    a1 = max(0.0, b1[2] - b1[0]) * max(0.0, b1[3] - b1[1])
    a2 = max(0.0, b2[2] - b2[0]) * max(0.0, b2[3] - b2[1])
    return inter / (a1 + a2 - inter) if (a1 + a2 - inter) > 0 else 0.0


def extract_pred_tables(doc: dict) -> list[tuple[tuple[float, float, float, float] | None, Node]]:
    out = []
    if is_custom_parser(doc):
        pw, ph = _custom_page_size(doc)
        for tbl in doc.get("tables") or []:
            bb = tbl.get("bbox")
            nb = (float(bb[0]) / pw, float(bb[1]) / ph, float(bb[2]) / pw, float(bb[3]) / ph) if (bb and len(bb) == 4 and pw > 0 and ph > 0) else None
            tree = html_to_tree(tbl.get("html") or "")
            out.append((nb, tree))
        return out

    pw, ph = _docling_page_size(doc)
    for tbl in doc.get("tables") or []:
        prov = (tbl.get("prov") or [None])[0]
        nb = None
        if prov and pw > 0 and ph > 0:
            bb = prov.get("bbox") or {}
            l, r = float(bb.get("l", 0)), float(bb.get("r", 0))
            t, b = float(bb.get("t", 0)), float(bb.get("b", 0))
            if str(bb.get("coord_origin", "BOTTOMLEFT")).upper() == "BOTTOMLEFT":
                y0, y1 = ph - max(t, b), ph - min(t, b)
            else:
                y0, y1 = min(t, b), max(t, b)
            nb = (min(l, r) / pw, y0 / ph, max(l, r) / pw, y1 / ph)
        out.append((nb, docling_table_to_tree(tbl)))
    return out


# =============================================================================
# LOAD GROUND TRUTH & RUN ALL 5 EVALUATIONS
# =============================================================================
t0 = perf_counter()
gt_raw = json.loads(GT_PATH.read_text(encoding="utf-8"))
gt_by_stem = {Path(p["page_info"]["image_path"]).stem: p for p in gt_raw}

# Use union of stems present in any system folder (matched against GT)
all_pred_stems = set()
for sys_name in SYSTEMS:
    sdir = BASE_DIR / sys_name
    for f in sdir.glob("*.json"):
        if f.name not in META_FILES and f.stem in gt_by_stem:
            all_pred_stems.add(f.stem)

all_stems = sorted(all_pred_stems)
pages_by_stem = {s: gt_by_stem[s]["page_info"] for s in all_stems}

# Precompute GT structures
gt_layout_lists = defaultdict(list)
present_cats = set()
doc_cat_images = defaultdict(list)
gt_text_blocks_map = {}
gt_text_str_map = {}
gt_tables_map = {}

for stem in all_stems:
    page = gt_by_stem[stem]
    pinfo = page["page_info"]
    pattr = pinfo.get("page_attribute") or {}
    pdf_type = pattr.get(PDF_TYPE_FIELD) or "unknown"
    doc_cat_images[pdf_type].append(stem)
    pw, ph = float(pinfo["width"]), float(pinfo["height"])

    for det in page.get("layout_dets") or ():
        mapped = OMNI_GT_TO_COMMON.get(det.get("category_type", ""))
        if mapped is None:
            continue
        poly = det.get("poly")
        if poly and len(poly) >= 8:
            xs = [float(poly[i]) for i in range(0, len(poly), 2)]
            ys = [float(poly[i]) for i in range(1, len(poly), 2)]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        elif det.get("bbox") and len(det["bbox"]) == 4:
            x, y, w, h = [float(v) for v in det["bbox"]]
            x1, y1, x2, y2 = x, y, x + w, y + h
        else:
            continue
        if x2 > x1 and y2 > y1:
            gt_layout_lists[(stem, mapped)].append([x1, y1, x2, y2])
            present_cats.add(mapped)

    gt_blks = extract_gt_ordered_text_blocks(page)
    gt_text_blocks_map[stem] = gt_blks
    gt_text_str_map[stem] = normalize_text("\n".join(t for _, _, _, t in gt_blks))

    gt_tbls = []
    for d in page.get("layout_dets") or ():
        if d.get("category_type") == "table" and not d.get("ignore") and (d.get("html") or "").strip():
            poly = d.get("poly") or [0] * 8
            xs, ys = poly[0::2], poly[1::2]
            nb = (min(xs) / pw, min(ys) / ph, max(xs) / pw, max(ys) / ph)
            tree = html_to_tree(d["html"])
            gt_tbls.append((nb, tree, is_complex_tree(tree)))
    if gt_tbls:
        gt_tables_map[stem] = gt_tbls

CAT_NAMES = sorted(present_cats)
doc_categories = sorted(doc_cat_images)
gt_layout_by_img_cat = {k: np.asarray(v, dtype=np.float64) for k, v in gt_layout_lists.items()}

# Evaluate each system
layout_overall = {}
layout_by_pdf_type = defaultdict(dict)
ocr_records = []
ro_records = []
table_records = []
perf_rows = []

for sys_name in SYSTEMS:
    sdir = BASE_DIR / sys_name
    pred_layout_by_img_cat = {}
    missing = 0

    for stem in all_stems:
        fpath = sdir / f"{stem}.json"
        pinfo = pages_by_stem[stem]
        pattr = pinfo.get("page_attribute") or {}
        pdf_type = pattr.get(PDF_TYPE_FIELD) or "unknown"
        layout_attr = pattr.get("layout") or "unknown"
        pw, ph = float(pinfo["width"]), float(pinfo["height"])

        if not fpath.exists():
            missing += 1
            doc = None
        else:
            doc = json.loads(fpath.read_text(encoding="utf-8"))

        # 1. Layout boxes
        if doc is not None:
            for cname, boxes in extract_pred_layout_boxes(doc, pw, ph).items():
                pred_layout_by_img_cat[(stem, cname)] = np.asarray(boxes, dtype=np.float64)

        # 2. Text / OCR
        gt_str = gt_text_str_map[stem]
        pred_blks = extract_pred_ordered_text_blocks(doc, pw, ph) if doc is not None else []
        if gt_str:
            pred_str = normalize_text("\n".join(t for _, t in pred_blks))
            ed = levenshtein_distance(pred_str, gt_str)
            ned = ed / max(len(pred_str), len(gt_str))
            ocr_records.append({
                "system": sys_name,
                "file": stem,
                "pdf_type": pdf_type,
                "layout": layout_attr,
                "gt_len": len(gt_str),
                "pred_len": len(pred_str),
                "ed": ed,
                "ned": ned,
                "sim": (1.0 - ned) * 100.0,
            })

        # 3. Reading Order
        ro_res = evaluate_reading_order_page(gt_text_blocks_map[stem], pred_blks)
        if ro_res is not None:
            ro_records.append({
                "system": sys_name,
                "file": stem,
                "pdf_type": pdf_type,
                "layout": layout_attr,
                **ro_res,
            })

        # 4. Table Structure
        gt_tbls = gt_tables_map.get(stem)
        if gt_tbls:
            pr_tbls = extract_pred_tables(doc) if doc is not None else []
            pairs = []
            for gi, (gb, _, _) in enumerate(gt_tbls):
                for pi, (pb, _) in enumerate(pr_tbls):
                    if pb is not None:
                        iou = box_iou(gb, pb)
                        if iou >= 0.1:
                            pairs.append((iou, gi, pi))
            pairs.sort(reverse=True)
            matched_g, matched_p = {}, set()
            for iou, gi, pi in pairs:
                if gi not in matched_g and pi not in matched_p:
                    matched_g[gi] = (pi, iou)
                    matched_p.add(pi)
            for gi, (_, gtree, is_cplx) in enumerate(gt_tbls):
                if gi in matched_g:
                    pi, iou = matched_g[gi]
                    ptree = pr_tbls[pi][1]
                    teds = compute_teds(ptree, gtree, struct_only=False) * 100.0
                    teds_s = compute_teds(ptree, gtree, struct_only=True) * 100.0
                    matched = True
                else:
                    teds, teds_s, iou, matched = 0.0, 0.0, 0.0, False
                table_records.append({
                    "system": sys_name,
                    "file": stem,
                    "pdf_type": pdf_type,
                    "complex": is_cplx,
                    "matched": matched,
                    "iou": iou,
                    "teds": teds,
                    "teds_struct": teds_s,
                })

    layout_overall[sys_name] = evaluate_layout_subset(all_stems, gt_layout_by_img_cat, pred_layout_by_img_cat, CAT_NAMES)
    for dcat in doc_categories:
        layout_by_pdf_type[dcat][sys_name] = evaluate_layout_subset(doc_cat_images[dcat], gt_layout_by_img_cat, pred_layout_by_img_cat, CAT_NAMES)

    # 5. Performance
    perf_path = sdir / "perf.json"
    err_path = sdir / "errors.json"
    p = json.loads(perf_path.read_text(encoding="utf-8")) if perf_path.exists() else {}
    errs = json.loads(err_path.read_text(encoding="utf-8")) if err_path.exists() else []
    perf_rows.append({
        "System": sys_name,
        "Passed": p.get("passed", len(all_stems) - missing),
        "Failed": p.get("failed", len(errs)),
        "Total Pages": p.get("total_pages", len(all_stems)),
        "Avg s/trang (1 GPU)": p.get("avg_seconds_per_page"),
        "Wall s/trang (2 GPU)": p.get("effective_seconds_per_page"),
        "Median s/file": p.get("median_seconds_per_file"),
        "P95 s/file": p.get("p95_seconds_per_file"),
        "Throughput (pg/s)": p.get("throughput_pages_per_sec"),
        "Peak RAM (MB/worker)": (p.get("peak_ram_mb") or {}).get("max_per_worker"),
        "Peak VRAM (MB/GPU)": (p.get("peak_vram_mb") or {}).get("max_per_gpu"),
    })

    ov = layout_overall[sys_name]
    print(f"✓ {sys_name}: missing={missing} | Layout F1@50={ov['F1_50']*100:.2f}% mAP={ov['mAP_50_95']*100:.2f}%")

df_ocr = pd.DataFrame(ocr_records)
df_ro = pd.DataFrame(ro_records)
df_tbl = pd.DataFrame(table_records)
df_perf = pd.DataFrame(perf_rows)
print(
    f"Hoàn tất đánh giá {len(all_stems)} trang ({len(CAT_NAMES)} layout classes, "
    f"{df_ocr['file'].nunique()} OCR pages, {df_ro['file'].nunique()} RO pages, "
    f"{len(df_tbl)//len(SYSTEMS)} tables) trong {perf_counter() - t0:.1f}s"
)

✓ default_docling: missing=0 | Layout F1@50=32.03% mAP=12.29%


✓ plugged_docling: missing=0 | Layout F1@50=51.72% mAP=35.45%


✓ custom_parser: missing=0 | Layout F1@50=45.81% mAP=21.57%
Hoàn tất đánh giá 518 trang (5 layout classes, 518 OCR pages, 518 RO pages, 406 tables) trong 29.3s


In [2]:
# Bảng tổng hợp đối đầu 5 yếu tố
summary_rows = []
for sys_name in SYSTEMS:
    ov = layout_overall[sys_name]
    o_sub = df_ocr[df_ocr["system"] == sys_name]
    r_sub = df_ro[df_ro["system"] == sys_name]
    r_valid = r_sub.dropna(subset=["tau", "nid"])
    t_sub = df_tbl[df_tbl["system"] == sys_name]
    t_m = t_sub[t_sub["matched"]]
    p_row = df_perf[df_perf["System"] == sys_name].iloc[0]
    summary_rows.append({
        "System": sys_name,
        "Layout F1@50 (↑)": f"{ov['F1_50'] * 100:.2f}%",
        "Layout mAP@50:95 (↑)": f"{ov['mAP_50_95'] * 100:.2f}%",
        "OCR NED (↓)": round(o_sub["ned"].mean(), 4),
        "OCR Sim % (↑)": round(o_sub["sim"].mean(), 2),
        "OCR Micro CER % (↓)": round(o_sub["ed"].sum() / o_sub["gt_len"].sum() * 100.0, 2),
        "RO Kendall Tau (↑)": round(r_valid["tau"].mean(), 4),
        "RO NID (↓)": round(r_valid["nid"].mean(), 4),
        "Table Match % (↑)": round(t_sub["matched"].mean() * 100.0, 2),
        "TEDS Matched % (↑)": round(t_m["teds"].mean(), 2) if len(t_m) else 0.0,
        "TEDS All % (↑)": round(t_sub["teds"].mean(), 2),
        "TEDS-Struct All % (↑)": round(t_sub["teds_struct"].mean(), 2),
        "Avg s/trang (↓)": round(float(p_row["Avg s/trang (1 GPU)"]), 2),
        "Peak RAM MB (↓)": round(float(p_row["Peak RAM (MB/worker)"]), 0),
        "Peak VRAM MB (↓)": round(float(p_row["Peak VRAM (MB/GPU)"]), 0),
    })

display(pd.DataFrame(summary_rows))

,System,Layout F1@50 (↑),Layout mAP@50:95 (↑),OCR NED (↓),OCR Sim % (↑),OCR Micro CER % (↓),RO Kendall Tau (↑),RO NID (↓),Table Match % (↑),TEDS Matched % (↑),TEDS All % (↑),TEDS-Struct All % (↑),Avg s/trang (↓),Peak RAM MB (↓),Peak VRAM MB (↓)
0,default_docling,32.03%,12.29%,0.9003,9.97,89.59,0.7865,0.0839,30.79,35.65,10.98,12.66,1.91,2315.0,2625.0
1,plugged_docling,51.72%,35.45%,0.3476,65.24,32.08,0.6446,0.2020,95.32,80.95,77.16,85.93,3.07,4032.0,10509.0
2,custom_parser,45.81%,21.57%,0.4486,55.14,43.55,0.7227,0.1575,67.49,81.82,55.22,61.08,30.73,2450.0,4323.0


In [3]:
def fmt_pct(val: float) -> str:
    return "-" if np.isnan(val) else f"{val * 100:.2f}%"

print("=== 1A. Layout Detection — Tổng hợp ===")
layout_rows = []
for s in SYSTEMS:
    r = layout_overall[s]
    layout_rows.append({
        "System": s,
        "mAP@50:95": fmt_pct(r["mAP_50_95"]),
        "AP@50": fmt_pct(r["AP_50"]),
        "AP@75": fmt_pct(r["AP_75"]),
        "P@50": fmt_pct(r["P_50"]),
        "R@50": fmt_pct(r["R_50"]),
        "F1@50": fmt_pct(r["F1_50"]),
        "F1@75": fmt_pct(r["F1_75"]),
    })
display(pd.DataFrame(layout_rows))

print("=== 1B. Layout Detection — F1@0.50 theo từng lớp nhãn ===")
cat_rows = []
for cname in CAT_NAMES:
    gt_cnt = layout_overall[SYSTEMS[0]]["per_cat_ap"][cname]["gt_count"]
    row = {"Class": cname, "GT Boxes": gt_cnt}
    for s in SYSTEMS:
        row[f"{s} F1@50"] = fmt_pct(layout_overall[s]["per_cat_f1_50"][cname])
    cat_rows.append(row)
display(pd.DataFrame(cat_rows))

print("=== 1C. Layout Detection — F1@0.50 theo loại PDF ===")
type_rows = []
for dcat in doc_categories:
    row = {"PDF Type": dcat, "Pages": len(doc_cat_images[dcat])}
    for s in SYSTEMS:
        row[s] = fmt_pct(layout_by_pdf_type[dcat][s]["F1_50"])
    type_rows.append(row)
display(pd.DataFrame(type_rows))

=== 1A. Layout Detection — Tổng hợp ===


,System,mAP@50:95,AP@50,AP@75,P@50,R@50,F1@50,F1@75
0,default_docling,12.29%,17.19%,13.27%,36.60%,28.48%,32.03%,24.32%
1,plugged_docling,35.45%,48.44%,37.26%,45.00%,60.78%,51.72%,41.08%
2,custom_parser,21.57%,30.81%,21.67%,51.03%,41.56%,45.81%,32.42%


=== 1B. Layout Detection — F1@0.50 theo từng lớp nhãn ===


,Class,GT Boxes,default_docling F1@50,plugged_docling F1@50,custom_parser F1@50
0,formula,107,3.33%,46.60%,10.69%
1,picture,532,34.64%,81.03%,49.96%
2,section_header,1543,56.34%,62.22%,61.81%
3,table,406,46.70%,92.87%,78.54%
4,text,7280,25.69%,46.22%,39.38%


=== 1C. Layout Detection — F1@0.50 theo loại PDF ===


,PDF Type,Pages,default_docling,plugged_docling,custom_parser
0,academic_paper,36,34.24%,59.62%,44.85%
1,book_vertical,55,22.52%,43.73%,39.00%
2,exam_paper,29,28.16%,60.87%,44.11%
3,handwritten_note,35,20.59%,47.58%,35.99%
4,lab_notebook,33,29.83%,51.67%,42.11%
5,magazine,37,33.90%,55.94%,47.77%
6,newspaper,28,40.28%,54.34%,56.49%
7,newspaper_vertical,21,23.53%,32.76%,41.80%
8,press_release,38,35.80%,57.01%,46.80%
9,report,42,27.16%,57.63%,42.28%


In [4]:
print("=== 2A. Text / OCR — Tổng hợp ===")
ocr_rows = []
for s in SYSTEMS:
    g = df_ocr[df_ocr["system"] == s]
    ocr_rows.append({
        "System": s,
        "Pages": len(g),
        "NED (↓)": round(g["ned"].mean(), 4),
        "Similarity % (↑)": round(g["sim"].mean(), 2),
        "Micro CER % (↓)": round(g["ed"].sum() / g["gt_len"].sum() * 100.0, 2),
        "Len Ratio (Pred/GT)": round(g["pred_len"].sum() / g["gt_len"].sum(), 3),
    })
display(pd.DataFrame(ocr_rows))

print("=== 2B. Text / OCR — Similarity (1-NED %) theo loại PDF ===")
piv_ocr = df_ocr.pivot_table(index="pdf_type", columns="system", values="sim", aggfunc="mean")[SYSTEMS].round(2)
piv_ocr.insert(0, "Pages", df_ocr[df_ocr["system"] == SYSTEMS[0]].groupby("pdf_type")["file"].count())
display(piv_ocr.sort_values("Pages", ascending=False))

if IS_JAPANESE:
    print("=== 2C. Text / OCR — In-Scope (Horizontal) vs Out-of-Scope (Vertical) ===")
    df_ocr["scope"] = df_ocr["layout"].apply(lambda x: "Out-of-Scope (Vertical)" if "vertical" in str(x).lower() else "In-Scope (Horizontal)")
    scope_rows = []
    for scope in ["In-Scope (Horizontal)", "Out-of-Scope (Vertical)"]:
        for s in SYSTEMS:
            g = df_ocr[(df_ocr["scope"] == scope) & (df_ocr["system"] == s)]
            scope_rows.append({
                "Scope": scope,
                "System": s,
                "Pages": len(g),
                "NED (↓)": round(g["ned"].mean(), 4),
                "Similarity % (↑)": round(g["sim"].mean(), 2),
                "Micro CER % (↓)": round(g["ed"].sum() / g["gt_len"].sum() * 100.0, 2),
            })
    display(pd.DataFrame(scope_rows))

=== 2A. Text / OCR — Tổng hợp ===


,System,Pages,NED (↓),Similarity % (↑),Micro CER % (↓),Len Ratio (Pred/GT)
0,default_docling,518,0.9003,9.97,89.59,0.128
1,plugged_docling,518,0.3476,65.24,32.08,0.927
2,custom_parser,518,0.4486,55.14,43.55,0.853


=== 2B. Text / OCR — Similarity (1-NED %) theo loại PDF ===


system,Pages,default_docling,plugged_docling,custom_parser
pdf_type,,,,
book_vertical,55,0.45,15.57,12.59
text_book,53,7.21,75.86,62.55
report,42,9.23,71.33,64.95
slide,38,14.32,68.60,51.19
press_release,38,12.63,75.01,64.39
magazine,37,13.66,69.71,63.68
user_manual,37,13.92,75.31,66.79
two_column_simple,36,24.44,80.00,64.14
academic_paper,36,4.07,73.27,65.91


=== 2C. Text / OCR — In-Scope (Horizontal) vs Out-of-Scope (Vertical) ===


,Scope,System,Pages,NED (↓),Similarity % (↑),Micro CER % (↓)
0,In-Scope (Horizontal),default_docling,442,0.8852,11.48,88.63
1,In-Scope (Horizontal),plugged_docling,442,0.2695,73.05,26.92
2,In-Scope (Horizontal),custom_parser,442,0.3817,61.83,39.27
3,Out-of-Scope (Vertical),default_docling,76,0.9876,1.24,98.36
4,Out-of-Scope (Vertical),plugged_docling,76,0.8023,19.77,78.93
5,Out-of-Scope (Vertical),custom_parser,76,0.8374,16.26,82.42


In [5]:
print("=== 3A. Reading Order — Tổng hợp ===")
ro_rows = []
for s in SYSTEMS:
    g = df_ro[df_ro["system"] == s]
    gv = g.dropna(subset=["tau", "nid"])
    ro_rows.append({
        "System": s,
        "Total RO Pages": len(g),
        "Scored Pages (≥2 matched)": len(gv),
        "Block Match % (↑)": round(g["matched"].sum() / g["gt_blocks"].sum() * 100.0, 2),
        "Kendall's Tau (↑)": round(gv["tau"].mean(), 4),
        "NID (↓)": round(gv["nid"].mean(), 4),
    })
display(pd.DataFrame(ro_rows))

print("=== 3B. Reading Order — Kendall's Tau & NID theo kiểu Layout ===")
ro_piv_tau = df_ro.dropna(subset=["tau"]).pivot_table(index="layout", columns="system", values="tau", aggfunc="mean")[SYSTEMS].round(4)
ro_piv_nid = df_ro.dropna(subset=["nid"]).pivot_table(index="layout", columns="system", values="nid", aggfunc="mean")[SYSTEMS].round(4)
ro_piv = pd.DataFrame({"Pages": df_ro[df_ro["system"] == SYSTEMS[0]].groupby("layout")["file"].count()})
for s in SYSTEMS:
    ro_piv[f"{s} Tau (↑)"] = ro_piv_tau[s]
for s in SYSTEMS:
    ro_piv[f"{s} NID (↓)"] = ro_piv_nid[s]
display(ro_piv.sort_values("Pages", ascending=False))

=== 3A. Reading Order — Tổng hợp ===


,System,Total RO Pages,Scored Pages (≥2 matched),Block Match % (↑),Kendall's Tau (↑),NID (↓)
0,default_docling,518,316,17.47,0.7865,0.0839
1,plugged_docling,518,516,81.46,0.6446,0.2020
2,custom_parser,518,469,66.38,0.7227,0.1575


=== 3B. Reading Order — Kendall's Tau & NID theo kiểu Layout ===


,Pages,default_docling Tau (↑),plugged_docling Tau (↑),custom_parser Tau (↑),default_docling NID (↓),plugged_docling NID (↓),custom_parser NID (↓)
layout,,,,,,,
book_vertical,55,-0.3579,-0.6891,-0.6377,0.4000,0.7157,0.6875
text_book,53,0.9695,0.8660,0.9282,0.0286,0.1131,0.0702
report,42,0.8213,0.8343,0.9107,0.0567,0.1344,0.0873
slide,38,0.7952,0.8900,0.9489,0.0893,0.1104,0.0625
press_release,38,0.8892,0.9193,0.8958,0.0516,0.0925,0.0984
magazine,37,0.9314,0.8319,0.9145,0.0468,0.1507,0.0711
user_manual,37,0.9450,0.8766,0.9149,0.0431,0.1187,0.0797
two_column_simple,36,0.8228,0.7272,0.8194,0.1059,0.1372,0.1313
academic_paper,36,0.9778,0.8644,0.8932,0.0167,0.1137,0.0896


In [6]:
print("=== 4A. Table Structure — Tổng hợp ===")
tbl_rows = []
for s in SYSTEMS:
    g = df_tbl[df_tbl["system"] == s]
    gm = g[g["matched"]]
    tbl_rows.append({
        "System": s,
        "GT Tables": len(g),
        "Matched": int(g["matched"].sum()),
        "Match % (↑)": round(g["matched"].mean() * 100.0, 2),
        "TEDS Matched % (↑)": round(gm["teds"].mean(), 2) if len(gm) else 0.0,
        "TEDS-Struct Matched % (↑)": round(gm["teds_struct"].mean(), 2) if len(gm) else 0.0,
        "TEDS All % (↑)": round(g["teds"].mean(), 2),
        "TEDS-Struct All % (↑)": round(g["teds_struct"].mean(), 2),
    })
display(pd.DataFrame(tbl_rows))

print("=== 4B. Table Structure — Simple vs Complex (Spanning) ===")
cplx_rows = []
for is_c, label in [(False, "Simple"), (True, "Complex (Spanning)")]:
    for s in SYSTEMS:
        g = df_tbl[(df_tbl["system"] == s) & (df_tbl["complex"] == is_c)]
        gm = g[g["matched"]]
        cplx_rows.append({
            "Complexity": label,
            "System": s,
            "GT Tables": len(g),
            "Match % (↑)": round(g["matched"].mean() * 100.0, 2),
            "TEDS Matched % (↑)": round(gm["teds"].mean(), 2) if len(gm) else 0.0,
            "TEDS-Struct Matched % (↑)": round(gm["teds_struct"].mean(), 2) if len(gm) else 0.0,
            "TEDS All % (↑)": round(g["teds"].mean(), 2),
        })
display(pd.DataFrame(cplx_rows))

=== 4A. Table Structure — Tổng hợp ===


,System,GT Tables,Matched,Match % (↑),TEDS Matched % (↑),TEDS-Struct Matched % (↑),TEDS All % (↑),TEDS-Struct All % (↑)
0,default_docling,406,125,30.79,35.65,41.12,10.98,12.66
1,plugged_docling,406,387,95.32,80.95,90.15,77.16,85.93
2,custom_parser,406,274,67.49,81.82,90.51,55.22,61.08


=== 4B. Table Structure — Simple vs Complex (Spanning) ===


,Complexity,System,GT Tables,Match % (↑),TEDS Matched % (↑),TEDS-Struct Matched % (↑),TEDS All % (↑)
0,Simple,default_docling,129,52.71,40.77,46.43,21.49
1,Simple,plugged_docling,129,94.57,85.90,92.43,81.24
2,Simple,custom_parser,129,82.17,85.65,93.66,70.38
3,Complex (Spanning),default_docling,277,20.58,29.54,34.79,6.08
4,Complex (Spanning),plugged_docling,277,95.67,78.67,89.10,75.26
5,Complex (Spanning),custom_parser,277,60.65,79.41,88.52,48.16


In [7]:
print("=== 5. Hiệu năng hệ thống (từ perf.json & errors.json — Kaggle 2x T4 GPU) ===")
display(df_perf)

=== 5. Hiệu năng hệ thống (từ perf.json & errors.json — Kaggle 2x T4 GPU) ===


,System,Passed,Failed,Total Pages,Avg s/trang (1 GPU),Wall s/trang (2 GPU),Median s/file,P95 s/file,Throughput (pg/s),Peak RAM (MB/worker),Peak VRAM (MB/GPU)
0,default_docling,518,0,518,1.9084,0.9882,1.6926,3.8807,1.0120,2314.93,2625.0
1,plugged_docling,518,0,518,3.0666,1.5664,2.9862,5.4850,0.6384,4031.96,10509.0
2,custom_parser,517,1,517,30.7343,15.4727,25.3075,67.4454,0.0646,2450.34,4323.0
